# CADP compression of the clean AIS

Trajectory compression keeps the points of a track that matter and drops the rest. This notebook
compresses **all** the clean AIS of the chosen dataset with **CADP**, the Context-Aware
Douglas-Peucker method of

> Mehri, S., Hooshangi, N., Mahdizadeh Gharakhanlou, N. (2025). *A Novel Context-Aware Douglas-Peucker (CADP) Trajectory Compression Method.* ISPRS Int. J. Geo-Inf. 14(2), 58. [doi:10.3390/ijgi14020058](https://doi.org/10.3390/ijgi14020058)

and saves the result next to that of notebook 03 (2stage-CPLS) in the same format, so the two can be compared.

The figures are interactive (hvplot / bokeh): they work in Jupyter and on [nbviewer](https://nbviewer.org/github/smehri/CATraj/blob/main/notebooks/04_cadp_compression.ipynb), but GitHub's notebook viewer does not run JavaScript, so they do not show on github.com.

1. **The method**: Douglas-Peucker, the context it is extended with, the parameters
2. **Load** the clean AIS split into trips, exactly as notebook 03 does
3. **Context**: the sea state (bottom depth, wave height, wave direction) annotated on every position
4. **Compress**: `compress` over every trip, CADP and plain Douglas-Peucker side by side
5. **Save**: `compressed_ais.parquet` and a summary table
6. **Results**: compression ratio overall and by vessel type; against Douglas-Peucker and 2stage-CPLS
7. **Vessels before and after**, and **how faithfully the context survived**

**Inputs**: `AIS/ais.parquet` and `Ocean/ocean_<month>.parquet` of the preprocessed folder (written
by `01_preprocessing.ipynb`), and the trips file of notebook 03 when it exists. **Outputs**:
`results/04_cadp_compression/<DATA>/`.
The package is built on [MovingPandas](https://movingpandas.org):

> Graser, A. (2019). *MovingPandas: Efficient Structures for Movement Data in Python.* GI_Forum 2019, 7(1), 54-68. [doi:10.1553/giscience2019_01_s54](https://doi.org/10.1553/giscience2019_01_s54)

## Data

The notebook compresses the preprocessed AIS (`ais.parquet`: cleaned, median-filtered positions of every
vessel, with speed, course, status and vessel type) and annotates it with the preprocessed **sea state**
(`ocean_<month>.parquet`: a 3-hourly WAVEWATCH III grid of bottom depth, wave height and wave direction).

By default it runs on the **committed, preprocessed two-day sample** (2015-11-09 to 2015-11-11 UTC, Brest
area, 565 vessels). To process the **full archive**, run `notebooks/01_preprocessing.ipynb` first (its
download and preprocessing sections), then set `DATA = "full"` below, or run the notebook with the
environment variable `CATRAJ_DATA=full`. The full archive covers October to December 2015 and February to
March 2016. **January 2016 is left out of every dataset**: the Zenodo archive has no sea state for it (its
`oc_january.csv` is a byte copy of `oc_november.csv`), so those AIS positions could not be annotated with
the context CADP needs and are not used anywhere.

Data: Ray, C. et al. (2019), *Heterogeneous integrated dataset for Maritime Intelligence, Surveillance, and
Reconnaissance*, Data in Brief 25, 104141, [doi:10.1016/j.dib.2019.104141](https://doi.org/10.1016/j.dib.2019.104141);
Zenodo [doi:10.5281/zenodo.1167595](https://doi.org/10.5281/zenodo.1167595),
record <https://zenodo.org/records/1167595>. AIS from NARI, Brest (CC BY-NC-SA 4.0: anything derived from it,
including this notebook's outputs, is shared under the same licence). Sea state: SHOM-IFREMER WAVEWATCH III
hindcast, citation mandatory: Boudiere, E. et al. (2013), *A suitable metocean hindcast database for the design
of Marine energy converters*, Int. J. Marine Energy 3-4, e40-e52,
[doi:10.1016/j.ijome.2013.11.010](https://doi.org/10.1016/j.ijome.2013.11.010).

In [1]:
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")   # see catraj/__init__.py

DATA = os.environ.get("CATRAJ_DATA", "sample")      # "sample" or "full"
N_JOBS = int(os.environ.get("CATRAJ_N_JOBS", "-1"))  # -1: every core; 1: serial

import time
from pathlib import Path

import holoviews as hv
import hvplot.pandas  # noqa: F401  -- registers .hvplot
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display
from pyproj import Transformer

hv.extension("bokeh")                  # interactive figures: zoom, pan, hover, click a legend entry

from catraj import parallel_map
from catraj.annotation import DLT, IDW, annotate_points
from catraj.compression import compress, compression_ratio, context_errors
from catraj.datasets import load_coastline, repo_root, sample_root
from catraj.preprocessing import split_trips

if DATA not in ("sample", "full"):
    raise ValueError(f"DATA must be 'sample' or 'full', not {DATA!r}")
ROOT = repo_root()
PRE = sample_root() if DATA == "sample" else ROOT / "data" / "zenodo_1167595" / "preprocessed"
OUT = ROOT / "results" / "04_cadp_compression" / DATA
OUT.mkdir(parents=True, exist_ok=True)
NB03 = ROOT / "results" / "03_two_stage_cpls_compression" / DATA     # notebook 03's outputs, if run

# ---- the parameters of CADP (the paper's experiment, Mehri et al. 2025) ---------
EPSILON = 1_000.0                    # Douglas-Peucker threshold, metres
THRESHOLDS = {                       # one per context variable, in its own units
    "wave_height": 0.15,             # m
    "wave_direction": 1.5,           # degrees, on the circle
    "bottom_depth": 80.0,            # m
}
CIRCULAR = ["wave_direction"]        # bearings: differenced on the circle
MODE = "post"                        # the method as published (see section 1)
# ---- the trips (identical to notebook 03) -----------------------------------------
MAX_GAP = 1_800.0        # a gap of over 30 minutes between two reports starts a new trip
MIN_TRIP_POINTS = 10     # trips with fewer positions are dropped
# ---- the comparison with the iterative mode: cheap on the sample, optional on the full archive
RUN_ITERATIVE = os.environ.get("CATRAJ_RUN_ITERATIVE", "1" if DATA == "sample" else "0") == "1"

print(f"DATA = {DATA!r}, N_JOBS = {N_JOBS}\ninput:  {PRE}\noutput: {OUT}")

DATA = 'full', N_JOBS = -1
input:  C:\dev\CATraj\data\zenodo_1167595\preprocessed
output: C:\dev\CATraj\results\04_cadp_compression\full


## Parallel processing

Three steps of this notebook run in parallel, all through the package's `n_jobs`
(`catraj.parallel_map` / `map_groups`, `src/catraj/_parallel.py`):

* **`annotate_points(..., n_jobs=N_JOBS)`** (section 3): the positions are grouped into *time blocks* of a few
  sea-state epochs, each with a small margin of epochs either side; the blocks are independent and are
  shared out over the processes.
* **`compress(..., by="trip_id", n_jobs=N_JOBS)`** (section 4, for CADP and for plain Douglas-Peucker):
  every trip is compressed independently of the others (Douglas-Peucker, the context errors and the union
  only ever look at one trip), so the trips are shared out. They are grouped into chunks of about equal
  number of positions, because the cost depends on a trip's length.
* **`split_trips(..., n_jobs=N_JOBS)`**, when the trips have to be recomputed (section 2), and the
  **context-fidelity check** (section 7, `catraj.parallel_map` over the trips): one task per trip.

`N_JOBS` follows the scikit-learn convention: `1` runs everything serially in this process, `-1` uses every
core, `k` uses `k` processes. The results come back in task order and every trip is computed by the same
code either way, so **parallel and serial runs give identical results**; only the wall time changes. Each
worker receives its share of the trips (pickled), so on the full archive the memory in use is roughly the
data frame in this process plus the chunks in flight in the workers; the annotation works one calendar
month at a time so the sea-state grid of the whole archive is never in memory. To run serially, set
`N_JOBS = 1` or start Jupyter with `CATRAJ_N_JOBS=1`.

## 1 · The method

**Douglas-Peucker** (DP) keeps the points that carry *spatial* information: a point is dropped when it lies
within `EPSILON` of the line between its neighbours. A point on a straight leg is spatially redundant, and yet
it may be the moment the waves rose or the water shoaled; once it is dropped, that information is gone from
every analysis downstream. **CADP** (Algorithm 2 of the paper) therefore runs in three steps:

1. **Spatial.** Simplify the trajectory with plain Douglas-Peucker at `EPSILON`.
2. **Context.** For every dropped point *i* between two consecutive retained points *a* and *b*, take the
   context the simplified trajectory implies at its time, the linear interpolation in *time*

   `c_hat_i = c_a + (c_b - c_a) * (t_i - t_a) / (t_b - t_a)`,   `e_i = | c_i - c_hat_i |`,

   and compare it with the context actually observed. If `e_i` exceeds that variable's threshold, point *i*
   is added back. This is the spatial error of Eq. 1 carried over to context: the projection onto the
   simplifying segment becomes the interpolation of the context along it, in the variable's own units.
3. **Union.** The result is the DP points plus every point added in step 2.

**Which context.** Here the context is the **sea state** met along the way, three **environmental**
variables, annotated on every position from the WAVEWATCH III grid (section 3):

| variable | meaning | threshold | note |
|---|---|---|---|
| `wave_height` | significant wave height | 0.15 m | |
| `wave_direction` | mean wave direction, a **bearing** | 1.5 degrees | differenced and interpolated on the circle: 350 and 10 degrees are 20 degrees apart |
| `bottom_depth` | bottom depth | 80 m | static in time; changes only as the vessel moves |

and `EPSILON` = 1,000 m, the values of the paper's experiment. Each variable is tested **on its own** and a
point is added if **any** of them is over its threshold. The package handles the other two domains of the paper
the same way (`thresholds` also takes *internal* variables such as `speed_over_ground`, and `nominal` takes
*semantic* ones such as the navigational status); this notebook keeps to the sea state, as the paper does.

**Two modes.** `mode="post"` is the method as published: every error is measured once, against the DP
trajectory, and every point over its threshold is added at once. That is the mode this notebook uses and saves.
`mode="iterative"` re-measures as the trajectory grows (only each segment's worst point is added per round), so
every dropped point ends up within its thresholds of its final neighbours, usually with far fewer points; the
notebook compares the two on the sample. `help(catraj.compression.cadp)` lists every decision.

**Filtering.** `compress` median-filters the positions and the threshold columns over 30 s by default. The
preprocessed AIS **already is** filtered (notebook 01 applies the same 30 s median) and the sea state is a
smooth 3-hourly field interpolated to the positions, so filtering again would only smooth smooth data, change the
positions later steps read and cost time. We pass `filter_window=None`: the compressed rows are then exactly
rows of the clean data, with their original values.

## 2 · Load the clean AIS as trips

The AIS is split into **trips** exactly as notebook 03 does it (`catraj.preprocessing.split_trips`): each
vessel's track is cut wherever two consecutive reports are **more than `MAX_GAP` = 30 minutes** apart (the
vessel was out of range, so a chord across the gap would invent a path), with no stay rule, and trips of fewer
than `MIN_TRIP_POINTS` = 10 positions are dropped. Every trip has a `trip_id`, unique over the dataset.

If notebook 03 has been run for this dataset, its saved `trips_ais.parquet` is read: that is the same
definition by construction and saves the split. Otherwise the trips are computed here with the same code and
the same two numbers.

In [2]:
COLUMNS = ["mmsi", "trip_id", "unix_time", "x", "y", "lon", "lat", "speed_over_ground",
           "course_over_ground", "vessel_type", "navigational_status", "true_heading",
           "rate_of_turn", "ship_type_code"]          # the columns of notebook 03's outputs

t0 = time.perf_counter()
saved_trips = NB03 / "trips_ais.parquet"
N_ALL = N_TRIPS_ALL = N_TINY = N_TINY_POINTS = None
if saved_trips.is_file():
    trips = pd.read_parquet(saved_trips)
    SOURCE = f"read from {saved_trips.relative_to(ROOT).as_posix()}"
    SPLIT_S = 0.0
else:
    from catraj.datasets import load_ais
    ais = load_ais(PRE)
    N_ALL = len(ais)
    ais = split_trips(ais, by="mmsi", time_column="unix_time", max_gap=MAX_GAP,
                      stay_radius=None, stay_duration=None, stay_column=None,
                      trip_column="trip", n_jobs=N_JOBS)
    sizes = ais.groupby(["mmsi", "trip"], sort=False).size()
    small = sizes < MIN_TRIP_POINTS
    N_TRIPS_ALL, N_TINY, N_TINY_POINTS = len(sizes), int(small.sum()), int(sizes[small].sum())
    big = sizes[~small].index.to_frame(index=False)
    trips = ais.merge(big, on=["mmsi", "trip"])
    trips = trips.sort_values(["mmsi", "unix_time"], kind="stable").reset_index(drop=True)
    trips["trip_id"] = trips.groupby(["mmsi", "trip"], sort=False).ngroup().astype("int64")
    del ais
    SPLIT_S = time.perf_counter() - t0
    SOURCE = "computed here with split_trips"
trips = pd.DataFrame(trips[COLUMNS]).sort_values(["mmsi", "unix_time"], kind="stable").reset_index(drop=True)
N = len(trips)
N_TRIPS = trips.trip_id.nunique()
print(f"trips {SOURCE}  ({time.perf_counter() - t0:.1f} s)")
if N_ALL is not None:
    print(f"split into {N_TRIPS_ALL:,} trips; dropped {N_TINY:,} tiny trips of fewer than {MIN_TRIP_POINTS} "
          f"positions ({N_TINY_POINTS:,} positions, {N_TINY_POINTS / N_ALL:.2%} of the data)")
print(f"{N_TRIPS:,} trips, {N:,} positions, {trips.mmsi.nunique():,} vessels, "
      f"{pd.to_datetime(trips.unix_time.min(), unit='s'):%Y-%m-%d} to "
      f"{pd.to_datetime(trips.unix_time.max(), unit='s'):%Y-%m-%d} UTC")
print("positions per trip:", trips.groupby("trip_id").size().describe().round(0).astype(int).to_dict())

trips read from results/03_two_stage_cpls_compression/full/trips_ais.parquet  (1.1 s)
19,407 trips, 14,777,418 positions, 2,076 vessels, 2015-10-01 to 2016-03-31 UTC
positions per trip: {'count': 19407, 'mean': 761, 'std': 4883, 'min': 10, '25%': 19, '50%': 48, '75%': 313, 'max': 192484}


## 3 · Context: the sea state at every position

CADP reads its context from ordinary columns, so the sea state is first **annotated** on the positions
(`catraj.annotation.annotate_points`, covered in `02_trajectory_annotation.ipynb`). The ocean grid is
3-hourly, in longitude and latitude (here projected to UTM 30N so that distances are in metres):

* **Spatially**, each variable is interpolated from the grid to the position by inverse-distance weighting
  over the 4 nearest cells (`IDW(k=4)`); for a bearing on the unit circle (`circular=True`), so that 350 and 10
  degrees average to 0, not 180.
* **Through time**, between the two 3-hourly epochs that bracket the position's time, linearly (`DLT`).
* **Bottom depth** does not change in time, so one epoch's grid is enough and only the spatial stage is applied.

The grid is stored one file per month, and millions of positions times many epochs would not fit in
memory at once, so the annotation works **one calendar month at a time**: the month's positions, and the grid
rows of the month plus a margin of two epochs, are read, annotated (with `n_jobs`) and released. The context
is already filtered by preprocessing (`filter_window=None`).

In [3]:
OCEAN_COLUMNS = ["unix_time", "lon", "lat", "bottom_depth", "wave_height", "wave_direction"]
MARGIN = 2 * 3 * 3600                      # grid rows read this far either side of a month
to_utm = Transformer.from_crs(4326, 32630, always_xy=True)

def read_ocean(lo, hi):
    # Grid rows with lo <= unix_time <= hi from every monthly file, cells projected to UTM 30N.
    files = sorted((PRE / "Ocean").glob("ocean_*.parquet"))
    parts = [pd.read_parquet(f, columns=OCEAN_COLUMNS,
                             filters=[("unix_time", ">=", lo), ("unix_time", "<=", hi)]) for f in files]
    oc = pd.concat([p for p in parts if len(p)], ignore_index=True)
    oc = oc.drop_duplicates(["unix_time", "lon", "lat"], ignore_index=True)   # months overlap by an epoch
    oc["x"], oc["y"] = to_utm.transform(oc.lon.to_numpy(), oc.lat.to_numpy())
    return oc

def stations(oc, column):
    # The table annotate_points takes: x, y, t, value.
    ok = oc[column].notna()
    return pd.DataFrame({"x": oc.x[ok].to_numpy(), "y": oc.y[ok].to_numpy(),
                         "t": oc.unix_time[ok].to_numpy(float), "value": oc[column][ok].to_numpy(float)})

month = pd.to_datetime(trips.unix_time, unit="s").dt.strftime("%Y-%m")
for name in THRESHOLDS:
    trips[name] = np.nan
rows = []
T0 = time.perf_counter()
for label in sorted(month.unique()):
    t0 = time.perf_counter()
    sel = np.flatnonzero((month == label).to_numpy())
    pts = trips.iloc[sel]
    x, y, t = pts.x.to_numpy(), pts.y.to_numpy(), pts.unix_time.to_numpy(float)
    oc = read_ocean(int(t.min()) - MARGIN, int(t.max()) + MARGIN)
    grid = oc[oc.unix_time == oc.unix_time.min()]            # depth: one epoch, spatial stage only
    grid = grid[grid.bottom_depth.notna()]
    values = {"bottom_depth": IDW(k=4).fit(grid.x, grid.y, grid.bottom_depth)(x, y)}
    for var in ("wave_height", "wave_direction"):
        circ = var in CIRCULAR
        values[var] = annotate_points(stations(oc, var), x, y, t,
                                      spatial_method=IDW(k=4, circular=circ),
                                      temporal_method=DLT(circular=circ), circular=circ,
                                      n_jobs=N_JOBS, filter_window=None)
    for var, v in values.items():
        trips.loc[trips.index[sel], var] = v
    rows.append({"month": label, "positions": len(sel), "epochs": oc.unix_time.nunique(),
                 **{f"{v}_annotated": float(np.isfinite(values[v]).mean()) for v in THRESHOLDS},
                 "seconds": time.perf_counter() - t0})
    del oc, grid
ANNOTATE_S = time.perf_counter() - T0
annotation = pd.DataFrame(rows)
print(f"annotated {N:,} positions in {ANNOTATE_S:.1f} s on {N_JOBS} job(s)")
annotation.round(4)

annotated 14,777,418 positions in 147.6 s on -1 job(s)


,month,positions,epochs,wave_height_annotated,wave_direction_annotated,bottom_depth_annotated,seconds
0,2015-10,3056341,250,1.0,1.0,1.0,28.7583
1,2015-11,3142703,244,1.0,1.0,1.0,28.3591
2,2015-12,2550998,248,1.0,1.0,1.0,27.6834
3,2016-02,3116542,234,1.0,1.0,1.0,30.2135
4,2016-03,2910834,249,1.0,1.0,1.0,32.4050


In [4]:
trips[list(THRESHOLDS)].describe().loc[["count", "mean", "std", "min", "max"]].round(2).T

,count,mean,std,min,max
wave_height,14777418.0,1.47,1.04,0.04,10.34
wave_direction,14777418.0,246.98,41.22,0.00,360.00
bottom_depth,14777418.0,36.51,156.17,2.58,4901.99


The `*_annotated` columns above are the share of positions that received a value (1.0 when the grid brackets
all of them); a position without a value is simply never added back on that variable's account. The table
shows the range the thresholds act on.

## 4 · Compress

`compress(traj, epsilon, thresholds, circular, time_column, by, mode, filter_window, n_jobs)` takes the
points of all trips as one `GeoDataFrame` in EPSG:32630, with `by="trip_id"` compressing every trip on its own,
and returns only the retained rows with every column intact. Without `thresholds` it is plain Douglas-Peucker,
so the same call gives the baseline; since CADP is DP *plus* added points, the DP rows are a subset of
the CADP rows, which tells us which points were kept **for context**.

The vessels are compressed type by type so that each vessel type has its own timing in the summary; within one
type the trips are shared out over `N_JOBS` processes. Seconds are wall time.

In [5]:
gdf = gpd.GeoDataFrame(trips, geometry=gpd.points_from_xy(trips.x, trips.y), crs=32630)

COMMON = dict(time_column="unix_time", by="trip_id", filter_window=None, n_jobs=N_JOBS)  # data already filtered
runs = {"DP": dict(thresholds=None),
        "CADP": dict(thresholds=THRESHOLDS, circular=CIRCULAR, mode=MODE)}
if RUN_ITERATIVE:
    runs["CADP iterative"] = dict(thresholds=THRESHOLDS, circular=CIRCULAR, mode="iterative")
KEY = {label: label.replace(" ", "_").lower() for label in runs}

kept, rows = {k: [] for k in runs}, []
T0 = time.perf_counter()
for vtype, part in gdf.groupby("vessel_type", sort=False):
    row = {"vessel_type": vtype, "vessels": part.mmsi.nunique(), "trips": part.trip_id.nunique(),
           "points_in": len(part)}
    for label, kw in runs.items():
        t0 = time.perf_counter()
        res = compress(part, EPSILON, **COMMON, **kw)
        row[f"{KEY[label]}_points_out"] = len(res)
        row[f"{KEY[label]}_compression_ratio_pct"] = compression_ratio(len(res), len(part))
        row[f"{KEY[label]}_seconds"] = time.perf_counter() - t0
        kept[label].append(res.index.to_numpy())
    rows.append(row)
COMPRESS_S = time.perf_counter() - T0
kept = {k: np.sort(np.concatenate(v)) for k, v in kept.items()}   # the index is trips' row number
del gdf
print(f"compressed {N:,} points in {COMPRESS_S:.1f} s on {N_JOBS} job(s)")
for label in runs:
    print(f"  {label:<15} keeps {len(kept[label]):>10,}   R = {compression_ratio(len(kept[label]), N):6.2f} %")

compressed 14,777,418 points in 72.2 s on -1 job(s)
  DP              keeps     71,165   R =  99.52 %
  CADP            keeps  9,301,789   R =  37.05 %


## 5 · Save

The compressed rows are rows of the clean data: no point is moved (unlike 2stage-CPLS, CADP has no stay
collapsing), so every column keeps its original value. Written under `results/04_cadp_compression/<DATA>/`:

* `compressed_ais.parquet`: the compressed points with the same columns as notebook 03's output
  (`mmsi`, `trip_id`, `unix_time`, `x`, `y`, `lon`, `lat`, `speed_over_ground`, `course_over_ground`,
  `vessel_type`, `navigational_status`, `true_heading`, `rate_of_turn`, `ship_type_code`; `trip_id` is the same
  as in notebook 03's `trips_ais.parquet`), plus the three context columns and `kept_by_context`, True on
  the points CADP added to what Douglas-Peucker keeps;
* `summary.csv`: points in and out, compression ratio and seconds, per vessel type and in total, for
  Douglas-Peucker and CADP (and CADP iterative when run);
* `context_fidelity.csv`: the check of section 7.

In [6]:
is_dp = np.zeros(N, bool); is_dp[kept["DP"]] = True
is_cadp = np.zeros(N, bool); is_cadp[kept["CADP"]] = True
compressed = trips.iloc[kept["CADP"]].copy()
compressed["kept_by_context"] = ~is_dp[kept["CADP"]]
compressed.to_parquet(OUT / "compressed_ais.parquet", index=False)

summary = pd.DataFrame(rows).sort_values("points_in", ascending=False)
total = {"vessel_type": "ALL", "vessels": trips.mmsi.nunique(), "trips": N_TRIPS, "points_in": N}
for label in runs:
    total[f"{KEY[label]}_points_out"] = len(kept[label])
    total[f"{KEY[label]}_compression_ratio_pct"] = compression_ratio(len(kept[label]), N)
    total[f"{KEY[label]}_seconds"] = summary[f"{KEY[label]}_seconds"].sum()
summary = pd.concat([summary, pd.DataFrame([total])], ignore_index=True)
summary["annotate_seconds"] = ANNOTATE_S      # the same for every row: one pass over all positions
summary["split_trips_seconds"] = SPLIT_S
summary["n_jobs"] = N_JOBS
summary.to_csv(OUT / "summary.csv", index=False)
for f in sorted(OUT.iterdir()):
    print(f"{f.name:24s} {f.stat().st_size / 1e6:8.2f} MB")

compressed_ais.parquet     459.94 MB
context_fidelity.csv         0.00 MB
summary.csv                  0.00 MB


## 6 · Results

**Compression ratio** R is the percentage of points removed (Eq. 9 of the paper):
`R = (1 - points out / points in) * 100`. First overall and by vessel type, for plain Douglas-Peucker and for
CADP, with the ratio of **2stage-CPLS** from notebook 03 next to them when that notebook has been run for this
dataset. The methods do different things: DP only drops points that lie near the line, CADP additionally keeps
every point where the sea state departs from its interpolation, and 2stage-CPLS keeps points for topology and
stays; so the ratios say what each criterion costs, not which one is better.

In [7]:
show = summary[["vessel_type", "vessels", "trips", "points_in", "dp_points_out", "dp_compression_ratio_pct",
                "cadp_points_out", "cadp_compression_ratio_pct", "cadp_seconds"]].copy()
show["added_for_context"] = show.cadp_points_out - show.dp_points_out
if (NB03 / "summary.csv").is_file():
    s3 = pd.read_csv(NB03 / "summary.csv")[["vessel_type", "points_in", "points_out", "compression_ratio_pct"]]
    s3 = s3.rename(columns={"points_in": "nb03_points_in", "points_out": "2stage_points_out",
                            "compression_ratio_pct": "2stage_cpls_ratio_pct"})
    show = show.merge(s3, on="vessel_type", how="left")
    same = bool((show.nb03_points_in == show.points_in).all())
    print("2stage-CPLS ratios read from", (NB03 / "summary.csv").relative_to(ROOT).as_posix(),
          "- same input positions:", same)
    show = show.drop(columns="nb03_points_in")
else:
    print("notebook 03 has not been run for this dataset: no 2stage-CPLS column")
show = show.round({"dp_compression_ratio_pct": 2, "cadp_compression_ratio_pct": 2, "cadp_seconds": 1,
                   "2stage_cpls_ratio_pct": 2})
print(f"overall: {N:,} -> {len(compressed):,} points with CADP, "
      f"R = {compression_ratio(len(compressed), N):.2f} %, {N / len(compressed):.1f} points in per point kept; "
      f"Douglas-Peucker alone {len(kept['DP']):,} (R = {compression_ratio(len(kept['DP']), N):.2f} %)")
n_added = int(compressed.kept_by_context.sum())
print(f"CADP added {n_added:,} points for context ({n_added / len(compressed):.1%} of those it keeps)")
show

2stage-CPLS ratios read from results/03_two_stage_cpls_compression/full/summary.csv - same input positions: True
overall: 14,777,418 -> 9,301,789 points with CADP, R = 37.05 %, 1.6 points in per point kept; Douglas-Peucker alone 71,165 (R = 99.52 %)
CADP added 9,230,624 points for context (99.2% of those it keeps)


,vessel_type,vessels,trips,points_in,dp_points_out,dp_compression_ratio_pct,cadp_points_out,cadp_compression_ratio_pct,cadp_seconds,added_for_context,2stage_points_out,2stage_cpls_ratio_pct
0,Passenger,38,1311,4045030,11296,99.72,2624103,35.13,8.2,2612807,442929,89.05
1,Fishing,262,7339,2866090,24954,99.13,1619596,43.49,7.6,1594642,445859,84.44
2,Unknown,235,1701,2189901,7753,99.65,1530011,30.13,3.9,1522258,204878,90.64
3,Cargo,904,4082,1252446,9650,99.23,700653,44.06,4.5,691003,239527,80.88
4,Tug,30,156,1213339,747,99.94,764634,36.98,2.1,763887,15824,98.70
5,Tanker,297,1038,626062,2753,99.56,449940,28.13,1.4,447187,69512,88.90
6,Other,54,765,594229,2606,99.56,395553,33.43,1.2,392947,40666,93.16
7,Military Ops,12,103,540227,603,99.89,380931,29.49,0.7,380328,20158,96.27
8,Pilot Vessel,5,445,308202,2354,99.24,90664,70.58,0.6,88310,66132,78.54
9,Sailing Vessel,164,1113,284513,4063,98.57,171159,39.84,0.8,167096,13756,95.17


In [8]:
types = summary[summary.vessel_type != "ALL"].sort_values("points_in", ascending=False).head(12)
cols = {"dp_compression_ratio_pct": "Douglas-Peucker", "cadp_compression_ratio_pct": "CADP"}
if "2stage_cpls_ratio_pct" in show:
    types = types.merge(show[["vessel_type", "2stage_cpls_ratio_pct"]], on="vessel_type", how="left")
    cols["2stage_cpls_ratio_pct"] = "2stage-CPLS (notebook 03)"
long = (types.rename(columns=cols).melt(id_vars="vessel_type", value_vars=list(cols.values()),
                                       var_name="method", value_name="R (%)"))
long["vessel_type"] = pd.Categorical(long.vessel_type, categories=types.vessel_type.tolist(), ordered=True)
long.hvplot.bar(x="vessel_type", y="R (%)", by="method", rot=35, width=900, height=380, ylim=(0, 100),
                color=["#8c959f", "#0969da", "#e16f24"][:len(cols)], legend="bottom",
                title="compression ratio of the 12 vessel types with most positions", xlabel="")

:Bars   [vessel_type,method]   (R (%))

**What to look at.** CADP's ratio can never exceed Douglas-Peucker's: every point it adds is a point DP
dropped. How far below it falls depends on how much sea state the vessel meets *relative to the thresholds*.
Wave height at 0.15 m and direction at 1.5 degrees are tight compared with the variation of a 3-hourly field
along a voyage, so on long tracks the interpolated context drifts past a threshold within a DP segment and many
points are added back (the next section shows this on single vessels). A vessel that hardly moves meets
hardly any change of context and gets few added points.

In [9]:
if RUN_ITERATIVE:
    order = [("Douglas-Peucker", "DP"), ("CADP (post, published)", "CADP"), ("CADP (iterative)", "CADP iterative")]
    allrow = summary[summary.vessel_type == "ALL"].iloc[0]
    cmp = pd.DataFrame({"method": [m for m, _ in order],
                        "points_out": [len(kept[k]) for _, k in order],
                        "R (%)": [compression_ratio(len(kept[k]), N) for _, k in order],
                        "seconds": [allrow[f"{KEY[k]}_seconds"] for _, k in order]})
    print("the same trips and thresholds, the two modes of CADP:")
    display(cmp.round(2))
else:
    print("CADP iterative not run (RUN_ITERATIVE = False)")

CADP iterative not run (RUN_ITERATIVE = False)


In `iterative` mode only each segment's worst point is added per round and the errors are measured again, so
it keeps the apex of a bow in the context curve and usually finds the other points on the bow now within
tolerance: far fewer points, and a bound on the final error (next section), at the cost of several rounds.
`post` is the published method and the one saved.

## 7 · Vessels before and after, and the fidelity of the context

**A few vessels.** For the four vessel types with most positions, the trip with the most positions among those
that span more than 20 km: the original track in grey, the points Douglas-Peucker keeps in orange, the points
CADP adds for context in blue, over the coastline. The lower panels show the wave height met along each trip
with the kept points on it. Zoom, pan and hover to read a point's time and wave height.

In [10]:
coast = load_coastline(PRE).to_crs(4326)
MOVING_KM = 20.0
MAX_PTS = 2_500                      # at most this many points per vessel and layer in a figure

top_types = summary[summary.vessel_type != "ALL"].vessel_type.head(4).tolist()
chosen = []
for vt in top_types:
    sizes_vt = trips[trips.vessel_type == vt].groupby("trip_id").size().sort_values(ascending=False)
    for tid in sizes_vt.index[:200]:
        g = trips[trips.trip_id == tid]
        if np.hypot(np.ptp(g.x), np.ptp(g.y)) / 1000 > MOVING_KM:
            chosen.append((vt, tid))
            break

def thin(df, n=MAX_PTS):
    return df.iloc[::max(1, len(df) // n)]

maps, series = [], []
for vt, tid in chosen:
    g = trips[trips.trip_id == tid].copy()
    g["time"] = pd.to_datetime(g.unix_time, unit="s")
    g["kept"] = np.where(is_dp[g.index], "Douglas-Peucker", np.where(is_cadp[g.index], "added for context", ""))
    dp, add = g[g.kept == "Douglas-Peucker"], g[g.kept == "added for context"]
    title = f"{vt}, vessel {g.mmsi.iloc[0]}: {len(g):,} points -> {len(dp) + len(add):,}"
    pad = 0.08
    box = coast.cx[g.lon.min() - pad:g.lon.max() + pad, g.lat.min() - pad:g.lat.max() + pad]
    m = thin(g).hvplot.line(x="lon", y="lat", geo=True, tiles="CartoLight", color="#8c959f", line_width=2,
                            title=title, width=430, height=380)
    if len(box):
        m = m * box.hvplot.polygons(geo=True, color="#ece8dc", line_color="#a8a39a", alpha=0.6)
    m = m * thin(add).hvplot.points(x="lon", y="lat", geo=True, color="#0969da", size=12,
                                    hover_cols=["time", "wave_height"], label="added for context")
    m = m * dp.hvplot.points(x="lon", y="lat", geo=True, color="#e16f24", size=30,
                             hover_cols=["time", "wave_height"], label="Douglas-Peucker")
    maps.append(m)
    s_ = (thin(g).hvplot.line(x="time", y="wave_height", color="#8c959f", width=430, height=220,
                              ylabel="wave height (m)")
          * thin(add).hvplot.scatter(x="time", y="wave_height", color="#0969da", size=18)
          * dp.hvplot.scatter(x="time", y="wave_height", color="#e16f24", size=30))
    series.append(s_)
print(f"{len(chosen)} vessels shown: orange Douglas-Peucker, blue added for context; "
      f"at most {MAX_PTS:,} points per layer are drawn")
hv.Layout(maps + series).cols(len(maps)).opts(shared_axes=False)

4 vessels shown: orange Douglas-Peucker, blue added for context; at most 2,500 points per layer are drawn


:Layout
   .Overlay.I    :Overlay
      .WMTS.I                              :WMTS   [Longitude,Latitude]
      .Curve.I                             :Curve   [lon]   (lat)
      .Polygons.I                          :Polygons   [Longitude,Latitude]
      .Points.Added_for_context            :Points   [lon,lat]   (time,wave_height)
      .Points.Douglas_hyphen_minus_Peucker :Points   [lon,lat]   (time,wave_height)
   .Overlay.II   :Overlay
      .WMTS.I                              :WMTS   [Longitude,Latitude]
      .Curve.I                             :Curve   [lon]   (lat)
      .Polygons.I                          :Polygons   [Longitude,Latitude]
      .Points.Added_for_context            :Points   [lon,lat]   (time,wave_height)
      .Points.Douglas_hyphen_minus_Peucker :Points   [lon,lat]   (time,wave_height)
   .Overlay.III  :Overlay
      .WMTS.I                              :WMTS   [Longitude,Latitude]
      .Curve.I                             :Curve   [lon]   (lat)
      .Polygons.I                          :Polygons   [Longitude,Latitude]
      .Points.Added_for_context            :Points   [lon,lat]   (time,wave_height)
      .Points.Douglas_hyphen_minus_Peucker :Points   [lon,lat]   (time,wave_height)
   .Overlay.IV   :Overlay
      .WMTS.I                              :WMTS   [Longitude,Latitude]
      .Curve.I                             :Curve   [lon]   (lat)
      .Polygons.I                          :Polygons   [Longitude,Latitude]
      .Points.Added_for_context            :Points   [lon,lat]   (time,wave_height)
      .Points.Douglas_hyphen_minus_Peucker :Points   [lon,lat]   (time,wave_height)
   .Overlay.V    :Overlay
      .Curve.I    :Curve   [time]   (wave_height)
      .Scatter.I  :Scatter   [time]   (wave_height)
      .Scatter.II :Scatter   [time]   (wave_height)
   .Overlay.VI   :Overlay
      .Curve.I    :Curve   [time]   (wave_height)
      .Scatter.I  :Scatter   [time]   (wave_height)
      .Scatter.II :Scatter   [time]   (wave_height)
   .Overlay.VII  :Overlay
      .Curve.I    :Curve   [time]   (wave_height)
      .Scatter.I  :Scatter   [time]   (wave_height)
      .Scatter.II :Scatter   [time]   (wave_height)
   .Overlay.VIII :Overlay
      .Curve.I    :Curve   [time]   (wave_height)
      .Scatter.I  :Scatter   [time]   (wave_height)
      .Scatter.II :Scatter   [time]   (wave_height)

**Fidelity.** CADP's promise is about the dropped points: how far is their context from the interpolation
between their *final* neighbours, the quantity the thresholds are about? `catraj.compression.context_errors`
measures it for every dropped point. In `post` mode the errors are measured against the **DP** neighbours, and
once points are added those are no longer the neighbours, so the bound holds on the DP segments, not on the final
result. The table gives the RMSE, the largest error and the share of dropped points over the threshold. Trips are
checked in parallel, one task each (on the full archive a random 3,000 trips, the same for every run, so the
check stays small).

In [11]:
rng = np.random.default_rng(0)
ids = trips.trip_id.unique()
if len(ids) > 3_000:
    ids = np.sort(rng.choice(ids, 3_000, replace=False))
sub = trips[trips.trip_id.isin(ids)]
keep_sets = {"Douglas-Peucker": is_dp, "CADP": is_cadp}
if RUN_ITERATIVE:
    keep_sets["CADP iterative"] = np.isin(np.arange(N), kept["CADP iterative"])
groups = [g for _, g in sub.groupby("trip_id", sort=False)]
tasks = [(g.unix_time.to_numpy(float), {v: g[v].to_numpy(float) for v in THRESHOLDS},
          {k: m[g.index] for k, m in keep_sets.items()}) for g in groups]
circular_vars = set(CIRCULAR)

def fidelity(task):
    t, ctx, masks = task
    out = {}
    for label, m in masks.items():
        for v, c in ctx.items():
            e = context_errors(t, c, m, circular=v in circular_vars)[~m]
            out[(label, v)] = e[np.isfinite(e)]
    return out

t0 = time.perf_counter()
per_trip = parallel_map(fidelity, tasks, N_JOBS, [len(t[0]) for t in tasks])
rows_f = []
for label in keep_sets:
    for v, thr in THRESHOLDS.items():
        e = np.concatenate([p[(label, v)] for p in per_trip])
        rows_f.append({"method": label, "variable": v, "threshold": thr,
                       "rmse": float(np.sqrt(np.mean(e ** 2))) if e.size else 0.0,
                       "max": float(e.max()) if e.size else 0.0,
                       "dropped_over_threshold_pct": 100 * float((e > thr).mean()) if e.size else 0.0})
fid = pd.DataFrame(rows_f)
fid.to_csv(OUT / "context_fidelity.csv", index=False)
print(f"{len(groups):,} trips, {len(sub):,} positions, checked in {time.perf_counter() - t0:.1f} s on {N_JOBS} job(s)")
fid.round(3)

3,000 trips, 2,205,603 positions, checked in 0.9 s on -1 job(s)


,method,variable,threshold,rmse,max,dropped_over_threshold_pct
0,Douglas-Peucker,wave_height,0.15,0.377,3.327,30.693
1,Douglas-Peucker,wave_direction,1.50,21.541,179.991,56.945
2,Douglas-Peucker,bottom_depth,80.00,20.470,1399.001,0.244
3,CADP,wave_height,0.15,0.032,0.576,0.704
4,CADP,wave_direction,1.50,0.748,35.882,4.823
5,CADP,bottom_depth,80.00,3.800,239.049,0.021


**Reading the table.** Douglas-Peucker ignores the context, so a share of the points it drops are over the
thresholds, by far more than the thresholds. CADP in `post` mode puts back every point over a threshold against
the DP neighbours, which cuts the RMSE and the share over the threshold; what remains is caused by the added
points changing the neighbours. `iterative` mode, when run, has a maximum within each threshold by construction.

## 8 · Summary

The cell below prints the headline numbers, which are also in `summary.csv`.

In [12]:
tot = summary[summary.vessel_type == "ALL"].iloc[0]
print(f"dataset: {DATA}   trips: {N_TRIPS:,}   positions in: {N:,}   vessels: {trips.mmsi.nunique():,}")
print(f"context: {', '.join(f'{k} {v:g}' for k, v in THRESHOLDS.items())}; epsilon {EPSILON:g} m; mode {MODE}")
print(f"Douglas-Peucker    : {int(tot.dp_points_out):>10,} points   R = {tot.dp_compression_ratio_pct:.2f} %")
print(f"CADP               : {int(tot.cadp_points_out):>10,} points   R = {tot.cadp_compression_ratio_pct:.2f} %")
if RUN_ITERATIVE:
    print(f"CADP iterative     : {int(tot.cadp_iterative_points_out):>10,} points   R = {tot.cadp_iterative_compression_ratio_pct:.2f} %")
if (NB03 / "summary.csv").is_file():
    s3 = pd.read_csv(NB03 / "summary.csv"); s3 = s3[s3.vessel_type == "ALL"].iloc[0]
    print(f"2stage-CPLS (nb 03): {int(s3.points_out):>10,} points   R = {s3.compression_ratio_pct:.2f} %")
print(f"seconds: annotate {ANNOTATE_S:.1f}, compress ({' + '.join(runs)}) {COMPRESS_S:.1f}, on {N_JOBS} job(s)")

dataset: full   trips: 19,407   positions in: 14,777,418   vessels: 2,076
context: wave_height 0.15, wave_direction 1.5, bottom_depth 80; epsilon 1000 m; mode post
Douglas-Peucker    :     71,165 points   R = 99.52 %
CADP               :  9,301,789 points   R = 37.05 %
2stage-CPLS (nb 03):  1,636,505 points   R = 88.93 %
seconds: annotate 147.6, compress (DP + CADP) 72.2, on -1 job(s)
